In [0]:
%sql
-- What does the current active pipeline look like — submitted but not yet issued/finaled?
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_active_pipeline AS
WITH status_buckets AS (
    SELECT 
        fp.permit_nbr,
        fp.status_desc,
        sd.full_date AS submitted_date,
        fp.status_date,
        CASE 
            WHEN fp.status_desc IN (
                'Permit Finshed', 'Permit Finaled', 'Issued', 'CofO Issued', 'Permit Closed', 
                'CofC Issued', 'Permit Expired', 'CofO Corrected', 'OK for CofC', 
                'Re-Activate Permit', 'Intent to Revoke', 'CofC Corrected', 'OK to Issue CofC', 
                'CofO Superseded', 'Order to Comply Issued', 'Not Required', 'PC Approved', 
                'CofO Reactivated', 'TCO Issued', 'Ready to Issue', 'Permit Extended', 
                'PC Info Complete', 'Refund Denied', 'Withdrawn', 'Cancelled', 'Revoked'
            ) THEN 'RESOLVED'
            WHEN fp.status_desc IN (
                'CofO in Progress', 'Refund in Progress', 'No Progress', 'Intent to Correct CofC', 
                'Plan Check', 'In Review', 'Submitted', 'Information Needed', 'Pending', ''
            ) THEN 'STUCK'
            ELSE 'STUCK' 
        END AS status_category
    FROM la_lakehouse.gold.fact_permits AS fp
    LEFT JOIN la_lakehouse.gold.dim_date AS sd
        ON fp.submitted_date_key = sd.date_key
    WHERE fp.submitted_date_key IS NOT NULL
    AND fp.status_desc IS NOT NULL
)
SELECT 
status_desc,
COUNT(*) AS total_permits,
ROUND(AVG(DATEDIFF(CURRENT_DATE(), submitted_date)),2) AS avg_days_active
FROM status_buckets
WHERE status_category = 'STUCK'
GROUP BY status_desc 